In [33]:
import datetime
import math
import re
import warnings
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import HTML, display

warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.float_format', '{:.4f}'.format)

LOGS_DIR = Path('../../logs')
REPORTS_DIR = Path('reports')
REPORTS_DIR.mkdir(exist_ok=True)

ARCH_CUTOFF = datetime.datetime(2026, 3, 28).timestamp()


def show(df, title='', filename=None):
    """Display a single full DataFrame as HTML."""
    html = df.to_html(index=True, border=1, na_rep='')
    if filename:
        path = REPORTS_DIR / filename
        path.write_text(f'<h2>{title}</h2>\n' + html)
        print(f'Saved → {path}')
    display(HTML((f'<h3>{title}</h3>' if title else '') + html))


def show_split(df_left, df_right, title_left='Old arch', title_right='New arch', filename=None):
    """Display two DataFrames side-by-side in scrollable panes."""
    left_html = df_left.to_html(index=True, border=1, na_rep='')
    right_html = df_right.to_html(index=True, border=1, na_rep='')
    html = f"""
    <div style="display:flex; gap:16px; align-items:flex-start;">
      <div style="flex:1; overflow-x:auto; overflow-y:auto; max-height:600px; border:1px solid #ccc; padding:6px;">
        <h4 style="margin:0 0 6px 0;">{title_left} ({len(df_left)} runs)</h4>
        {left_html}
      </div>
      <div style="flex:1; overflow-x:auto; overflow-y:auto; max-height:600px; border:1px solid #ccc; padding:6px;">
        <h4 style="margin:0 0 6px 0;">{title_right} ({len(df_right)} runs)</h4>
        {right_html}
      </div>
    </div>
    """
    if filename:
        path = REPORTS_DIR / filename
        path.write_text(html)
        print(f'Saved → {path}')
    display(HTML(html))

In [34]:
def strip_omegaconf(text):
    pattern = re.compile(r'\$\{[^{}]+\}')
    for _ in range(10):
        prev = text
        text = pattern.sub('null', text)
        if text == prev:
            break
    return text


def extract_cls_name(raw_text, key):
    m = re.search(rf'^{re.escape(key)}:\s*\$\{{cls:([^}}]+)\}}', raw_text, re.MULTILINE)
    return m.group(1).split('.')[-1] if m else ''


def load_config(config_path):
    try:
        raw = config_path.read_text()
        parsed = yaml.safe_load(strip_omegaconf(raw)) or {}
        return parsed, raw
    except Exception:
        return {}, ''


def load_eval_csv(eval_path):
    try:
        df = pd.read_csv(eval_path)
        return df if not df.empty and 'eval/success_rate' in df.columns else None
    except Exception:
        return None


def _flatten(d, prefix):
    if not d or not isinstance(d, dict):
        return {}
    return {f'{prefix}/{k}': v for k, v in d.items()}


def _safe(val):
    try:
        v = val.item() if hasattr(val, 'item') else val
        return None if (isinstance(v, float) and math.isnan(v)) else v
    except Exception:
        return None


def get_run_start_ts(run_dir):
    mtimes = [f.stat().st_mtime for f in run_dir.rglob('*') if f.is_file()]
    return min(mtimes) if mtimes else 0.0


def _extract_run(run_dir, run_type):
    eval_df = load_eval_csv(run_dir / 'eval.csv')
    if eval_df is None:
        return None
    cfg, raw = load_config(run_dir / 'config.yaml')
    best_row = eval_df.loc[eval_df['eval/success_rate'].idxmax()]
    last_row = eval_df.iloc[-1]
    fc = 'collected_frames'
    ts = get_run_start_ts(run_dir)
    return {
        'run_id': run_dir.name,
        'run_type': run_type,
        'arch': 'new' if ts >= ARCH_CUTOFF else 'old',
        'run_date': datetime.datetime.fromtimestamp(ts).strftime('%Y-%m-%d'),
        'path': str(run_dir),
        'best_success_rate': _safe(best_row['eval/success_rate']),
        'last_success_rate': _safe(last_row['eval/success_rate']),
        'best_mean_reward': _safe(best_row.get('eval/mean_step_reward')),
        'best_at_frames': _safe(best_row.get(fc)),
        'total_frames': _safe(eval_df[fc].max()) if fc in eval_df.columns else None,
        'num_evals': len(eval_df),
        'name': cfg.get('name', ''),
        'comment': str(cfg.get('comment', ''))[:80],
        'num_robots': cfg.get('num_robots'),
        'terrain': cfg.get('terrain'),
        'task': cfg.get('task'),
        'seed': cfg.get('seed'),
        'scheduler': extract_cls_name(raw, 'scheduler'),
        **_flatten(cfg.get('scheduler_opts', {}), 'sched'),
        'gamma': cfg.get('gae_opts', {}).get('gamma'),
        'lmbda': cfg.get('gae_opts', {}).get('lmbda'),
        'clip_epsilon': cfg.get('ppo_opts', {}).get('clip_epsilon'),
        'entropy_coef': cfg.get('ppo_opts', {}).get('entropy_coef'),
        **_flatten(cfg.get('env_cfg_overrides', {}), 'env'),
    }


def collect_runs(logs_dir):
    runs = []
    skip = ('eval_', 'host_', 'isaac_', 'wandb')
    for entry in sorted(logs_dir.iterdir()):
        if not entry.is_dir() or any(entry.name.startswith(p) for p in skip):
            continue
        if entry.name.startswith('optuna_'):
            for trial in sorted(entry.iterdir()):
                if trial.is_dir() and re.match(r'.+_\d+$', trial.name):
                    r = _extract_run(trial, 'optuna')
                    if r:
                        runs.append(r)
        elif entry.name.startswith('train_'):
            r = _extract_run(entry, 'train')
            if r:
                runs.append(r)
    return runs


print('Loading runs...')
runs = collect_runs(LOGS_DIR)
df = pd.DataFrame(runs).sort_values('last_success_rate', ascending=False).reset_index(drop=True)

df_old = df[df.arch == 'old'].reset_index(drop=True)
df_new = df[df.arch == 'new'].reset_index(drop=True)

env_cols = sorted(c for c in df.columns if c.startswith('env/'))
sched_cols = sorted(c for c in df.columns if c.startswith('sched/'))
ppo_cols = [c for c in ('scheduler', 'gamma', 'lmbda', 'clip_epsilon', 'entropy_coef') if c in df.columns]

print(f'Loaded {len(df)} runs — old arch: {len(df_old)}, new arch: {len(df_new)}')

Loading runs...
Loaded 608 runs — old arch: 335, new arch: 273


## Summary — old vs new architecture

In [35]:
summary_cols = [
    'run_id', 'run_date', 'run_type', 'best_success_rate', 'last_success_rate',
    'best_mean_reward', 'best_at_frames', 'total_frames', 'num_evals',
    'num_robots', 'terrain', 'name', 'comment'
]
summary_cols = [c for c in summary_cols if c in df.columns]
show_split(df_old[summary_cols], df_new[summary_cols],
           title_left='Old arch (pre 2026-03-28)',
           title_right='New arch (post 2026-03-28)',
           filename='summary_split.html')

Saved → reports/summary_split.html


,run_id,run_date,run_type,best_success_rate,last_success_rate,best_mean_reward,best_at_frames,total_frames,num_evals,num_robots,terrain,name,comment
0,optuna_ftr_10724353_89,2026-03-23,optuna,0.9231,0.9231,6.3876,5177344.0000,5177344.0000,63,128,cur_mixed,ftr_ppo_crossing_potential,"Best Optuna trial (v3 study, trial #28, 88.9% success rate). Trained with 128 en"
1,optuna_ftr_10724353_101,2026-03-23,optuna,0.9211,0.9211,4.8529,5177344.0000,5177344.0000,63,128,cur_mixed,ftr_ppo_crossing_potential,"Best Optuna trial (v3 study, trial #28, 88.9% success rate). Trained with 128 en"
2,optuna_ftr_10724353_88,2026-03-23,optuna,0.9242,0.9028,6.6224,4931584.0000,5177344.0000,63,128,cur_mixed,ftr_ppo_crossing_potential,"Best Optuna trial (v3 study, trial #28, 88.9% success rate). Trained with 128 en"
3,optuna_ftr_10724353_90,2026-03-23,optuna,0.8704,0.8704,5.6933,3538944.0000,3538944.0000,43,128,cur_mixed,ftr_ppo_crossing_potential,"Best Optuna trial (v3 study, trial #28, 88.9% success rate). Trained with 128 en"
4,optuna_ftr_10718414_79,2026-03-22,optuna,0.9000,0.8696,6.4739,2473984.0000,5177344.0000,63,128,cur_mixed,ftr_ppo_crossing_potential,"Best Optuna trial (v1 study, trial #100, 41.4% success rate). Trained with 128 e"
5,optuna_ftr_10725954_131,2026-03-23,optuna,0.9677,0.8689,4.9837,3620864.0000,5177344.0000,63,128,cur_mixed,ftr_ppo_crossing_potential,"Best Optuna trial (v3 study, trial #28, 88.9% success rate). Trained with 128 en"
6,optuna_ftr_10724353_84,2026-03-23,optuna,0.9067,0.8681,7.5068,4521984.0000,5177344.0000,63,128,cur_mixed,ftr_ppo_crossing_potential,"Best Optuna trial (v3 study, trial #28, 88.9% success rate). Trained with 128 en"
7,optuna_ftr_10718414_28,2026-03-22,optuna,0.9722,0.8519,4.7076,4276224.0000,5177344.0000,63,128,cur_mixed,ftr_ppo_crossing_potential,"Best Optuna trial (v1 study, trial #100, 41.4% success rate). Trained with 128 e"
8,optuna_ftr_10724353_123,2026-03-23,optuna,0.9206,0.8491,6.8567,5013504.0000,5177344.0000,63,128,cur_mixed,ftr_ppo_crossing_potential,"Best Optuna trial (v3 study, trial #28, 88.9% success rate). Trained with 128 en"
9,optuna_ftr_10718414_20,2026-03-22,optuna,1.0000,0.8333,6.6649,2473984.0000,5177344.0000,63,128,cur_mixed,ftr_ppo_crossing_potential,"Best Optuna trial (v1 study, trial #100, 41.4% success rate). Trained with 128 e"


## env_cfg_overrides — old vs new architecture

In [39]:
env_display = ['run_id', 'run_date', 'best_success_rate', 'last_success_rate'] + env_cols
print(f'{len(env_cols)} env_cfg_overrides fields: {env_cols}')
show_split(df_old[env_display], df_new[env_display],
           title_left='env_cfg_overrides — old arch',
           title_right='env_cfg_overrides — new arch',
           filename='env_split.html')

34 env_cfg_overrides fields: ['env/action_bonus_coef', 'env/clearance_coef', 'env/clearance_threshold', 'env/failed_reward', 'env/flipper_contact_coef', 'env/flipper_contact_threshold', 'env/flipper_pos_max_deg', 'env/goal_reached_reward', 'env/joint_angle_variance_coef', 'env/joint_vel_variance_coef', 'env/lin_action_ratio', 'env/lin_to_flipper_action_ratio', 'env/pitch_coef', 'env/pitch_rate_coef', 'env/potential_coef', 'env/potential_failed_reward', 'env/potential_gamma', 'env/potential_goal_reached_reward', 'env/potential_joint_angle_variance_coef', 'env/potential_joint_vel_variance_coef', 'env/potential_pitch_coef', 'env/potential_pitch_rate_coef', 'env/potential_roll_coef', 'env/potential_roll_rate_coef', 'env/potential_step_penalty', 'env/roll_coef', 'env/roll_rate_coef', 'env/shaping_coef', 'env/shaping_gamma', 'env/shock_coef', 'env/shock_scale', 'env/spawn_yaw_range', 'env/step_penalty', 'env/timeout_penalty']
Saved → reports/env_split.html


,run_id,run_date,best_success_rate,last_success_rate,env/action_bonus_coef,env/clearance_coef,env/clearance_threshold,env/failed_reward,env/flipper_contact_coef,env/flipper_contact_threshold,env/flipper_pos_max_deg,env/goal_reached_reward,env/joint_angle_variance_coef,env/joint_vel_variance_coef,env/lin_action_ratio,env/lin_to_flipper_action_ratio,env/pitch_coef,env/pitch_rate_coef,env/potential_coef,env/potential_failed_reward,env/potential_gamma,env/potential_goal_reached_reward,env/potential_joint_angle_variance_coef,env/potential_joint_vel_variance_coef,env/potential_pitch_coef,env/potential_pitch_rate_coef,env/potential_roll_coef,env/potential_roll_rate_coef,env/potential_step_penalty,env/roll_coef,env/roll_rate_coef,env/shaping_coef,env/shaping_gamma,env/shock_coef,env/shock_scale,env/spawn_yaw_range,env/step_penalty,env/timeout_penalty
0,optuna_ftr_10724353_89,2026-03-23,0.9231,0.9231,,,,-17.9192,,,90.0000,24.7999,1.4966,3.0788,,,1.6317,0.5960,,,,,,,,,,,,0.4335,0.3807,49.4819,,,,,-0.0523,
1,optuna_ftr_10724353_101,2026-03-23,0.9211,0.9211,,,,-9.9411,,,90.0000,10.1156,-0.1256,3.1608,,,1.6817,0.3962,,,,,,,,,,,,0.2767,0.4189,6.0685,,,,,-0.2282,
2,optuna_ftr_10724353_88,2026-03-23,0.9242,0.9028,,,,-9.2488,,,90.0000,16.9994,0.4180,0.8505,,,1.9962,0.4781,,,,,,,,,,,,0.8033,0.6544,16.8321,,,,,-0.3735,
3,optuna_ftr_10724353_90,2026-03-23,0.8704,0.8704,,,,-9.7566,,,90.0000,21.9994,-0.1773,1.9227,,,0.9209,0.5411,,,,,,,,,,,,0.2571,0.5308,30.5828,,,,,-0.6497,
4,optuna_ftr_10718414_79,2026-03-22,0.9000,0.8696,,,,-11.3228,,,90.0000,20.0341,0.2933,2.2303,,,1.7907,0.2051,,,,,,,,,,,,0.5423,0.3932,23.0016,,,,,-0.5062,
5,optuna_ftr_10725954_131,2026-03-23,0.9677,0.8689,,,,-4.2264,,,90.0000,15.6442,0.8547,0.2710,,,1.9899,0.9263,,,,,,,,,,,,0.8110,1.7067,12.0560,,,,,-0.4988,
6,optuna_ftr_10724353_84,2026-03-23,0.9067,0.8681,,,,-10.4841,,,90.0000,29.2913,-0.9021,-0.4074,,,1.6744,0.4908,,,,,,,,,,,,1.1302,0.5000,37.3932,,,,,-0.2306,
7,optuna_ftr_10718414_28,2026-03-22,0.9722,0.8519,,,,-19.4021,,,90.0000,29.4114,-1.9788,1.4934,,,0.0239,1.7444,,,,,,,,,,,,0.8720,0.3501,42.1487,,,,,-0.0880,
8,optuna_ftr_10724353_123,2026-03-23,0.9206,0.8491,,,,-3.5043,,,90.0000,12.2104,0.8517,0.5585,,,1.9169,0.8579,,,,,,,,,,,,0.6162,1.2109,9.0775,,,,,-0.4520,
9,optuna_ftr_10718414_20,2026-03-22,1.0000,0.8333,,,,-8.3200,,,90.0000,29.1568,0.6401,1.9464,,,1.9313,0.3491,,,,,,,,,,,,0.5698,0.1039,30.4355,,,,,-0.1569,


## Scheduler & PPO — old vs new architecture

In [ ]:
sched_display = ['run_id', 'run_date', 'best_success_rate'] + ppo_cols + sched_cols
show_split(df_old[sched_display], df_new[sched_display],
           title_left='Scheduler & PPO — old arch',
           title_right='Scheduler & PPO — new arch',
           filename='sched_split.html')

Saved → reports/sched_split.html


,run_id,run_date,best_success_rate,scheduler,gamma,lmbda,clip_epsilon,entropy_coef,sched/end_factor,sched/start_factor,sched/total_iters
0,optuna_ftr_10724353_89,2026-03-23,0.9231,LinearLR,0.9987,0.9512,0.3266,0.0350,0.2432,1.0000,None
1,optuna_ftr_10724353_101,2026-03-23,0.9211,LinearLR,0.9916,0.9282,0.2417,0.0350,0.2432,1.0000,None
2,optuna_ftr_10724353_88,2026-03-23,0.9242,LinearLR,0.9931,0.9463,0.2699,0.0350,0.2432,1.0000,None
3,optuna_ftr_10724353_90,2026-03-23,0.8704,LinearLR,0.9981,0.9534,0.3217,0.0350,0.2432,1.0000,None
4,optuna_ftr_10718414_79,2026-03-22,0.9000,LinearLR,0.9944,0.9546,0.2593,0.0350,0.2432,1.0000,None
5,optuna_ftr_10725954_131,2026-03-23,0.9677,LinearLR,0.9955,0.9461,0.3333,0.0350,0.2432,1.0000,None
6,optuna_ftr_10724353_84,2026-03-23,0.9067,LinearLR,0.9969,0.9770,0.2830,0.0350,0.2432,1.0000,None
7,optuna_ftr_10718414_28,2026-03-22,0.9722,LinearLR,0.9907,0.9308,0.3050,0.0350,0.2432,1.0000,None
8,optuna_ftr_10724353_123,2026-03-23,0.9206,LinearLR,0.9920,0.9600,0.3008,0.0350,0.2432,1.0000,None
9,optuna_ftr_10718414_20,2026-03-22,1.0000,LinearLR,0.9917,0.9445,0.2436,0.0350,0.2432,1.0000,None


## Top 20 — old vs new architecture (full config)

In [38]:
detail_cols = [
    'run_id', 'run_date', 'run_type', 'best_success_rate', 'last_success_rate',
    'total_frames', 'num_robots', 'terrain', 'seed', 'name', 'comment'
] + ppo_cols + sched_cols + env_cols
detail_cols = [c for c in detail_cols if c in df.columns]

top_old = df_old.head(20)[detail_cols].set_index('run_id').T
top_new = df_new.head(20)[detail_cols].set_index('run_id').T
show_split(top_old, top_new,
           title_left='Top 20 — old arch',
           title_right='Top 20 — new arch',
           filename='top20_split.html')

Saved → reports/top20_split.html


run_id,optuna_ftr_10724353_89,optuna_ftr_10724353_101,optuna_ftr_10724353_88,optuna_ftr_10724353_90,optuna_ftr_10718414_79,optuna_ftr_10725954_131,optuna_ftr_10724353_84,optuna_ftr_10718414_28,optuna_ftr_10724353_123,optuna_ftr_10718414_20,optuna_ftr_10718414_66,optuna_ftr_10718414_37,optuna_ftr_10718414_64,optuna_ftr_10724353_100,optuna_ftr_10724353_85,optuna_ftr_10718414_70,optuna_ftr_10718414_82,optuna_ftr_10724353_97,optuna_ftr_10724353_118,optuna_ftr_10724353_117
run_date,2026-03-23,2026-03-23,2026-03-23,2026-03-23,2026-03-22,2026-03-23,2026-03-23,2026-03-22,2026-03-23,2026-03-22,2026-03-22,2026-03-22,2026-03-22,2026-03-23,2026-03-23,2026-03-22,2026-03-22,2026-03-23,2026-03-23,2026-03-23
run_type,optuna,optuna,optuna,optuna,optuna,optuna,optuna,optuna,optuna,optuna,optuna,optuna,optuna,optuna,optuna,optuna,optuna,optuna,optuna,optuna
best_success_rate,0.9231,0.9211,0.9242,0.8704,0.9000,0.9677,0.9067,0.9722,0.9206,1.0000,0.8462,0.9219,0.8913,0.8681,0.8953,0.8205,0.8493,0.8824,0.8261,0.8491
last_success_rate,0.9231,0.9211,0.9028,0.8704,0.8696,0.8689,0.8681,0.8519,0.8491,0.8333,0.8333,0.8308,0.8302,0.8182,0.8182,0.8173,0.8140,0.8118,0.8116,0.8056
total_frames,5177344.0000,5177344.0000,5177344.0000,3538944.0000,5177344.0000,5177344.0000,5177344.0000,5177344.0000,5177344.0000,5177344.0000,5177344.0000,5177344.0000,5177344.0000,5177344.0000,5177344.0000,5177344.0000,5177344.0000,5177344.0000,5177344.0000,5177344.0000
num_robots,128,128,128,128,128,128,128,128,128,128,128,128,128,128,128,128,128,128,128,128
terrain,cur_mixed,cur_mixed,cur_mixed,cur_mixed,cur_mixed,cur_mixed,cur_mixed,cur_mixed,cur_mixed,cur_mixed,cur_mixed,cur_mixed,cur_mixed,cur_mixed,cur_mixed,cur_mixed,cur_mixed,cur_mixed,cur_mixed,cur_mixed
seed,42,42,42,42,42,42,42,42,42,42,42,42,42,42,42,42,42,42,42,42
name,ftr_ppo_crossing_potential,ftr_ppo_crossing_potential,ftr_ppo_crossing_potential,ftr_ppo_crossing_potential,ftr_ppo_crossing_potential,ftr_ppo_crossing_potential,ftr_ppo_crossing_potential,ftr_ppo_crossing_potential,ftr_ppo_crossing_potential,ftr_ppo_crossing_potential,ftr_ppo_crossing_potential,ftr_ppo_crossing_potential,ftr_ppo_crossing_potential,ftr_ppo_crossing_potential,ftr_ppo_crossing_potential,ftr_ppo_crossing_potential,ftr_ppo_crossing_potential,ftr_ppo_crossing_potential,ftr_ppo_crossing_potential,ftr_ppo_crossing_potential
comment,"Best Optuna trial (v3 study, trial #28, 88.9% success rate). Trained with 128 en","Best Optuna trial (v3 study, trial #28, 88.9% success rate). Trained with 128 en","Best Optuna trial (v3 study, trial #28, 88.9% success rate). Trained with 128 en","Best Optuna trial (v3 study, trial #28, 88.9% success rate). Trained with 128 en","Best Optuna trial (v1 study, trial #100, 41.4% success rate). Trained with 128 e","Best Optuna trial (v3 study, trial #28, 88.9% success rate). Trained with 128 en","Best Optuna trial (v3 study, trial #28, 88.9% success rate). Trained with 128 en","Best Optuna trial (v1 study, trial #100, 41.4% success rate). Trained with 128 e","Best Optuna trial (v3 study, trial #28, 88.9% success rate). Trained with 128 en","Best Optuna trial (v1 study, trial #100, 41.4% success rate). Trained with 128 e","Best Optuna trial (v1 study, trial #100, 41.4% success rate). Trained with 128 e","Best Optuna trial (v1 study, trial #100, 41.4% success rate). Trained with 128 e","Best Optuna trial (v1 study, trial #100, 41.4% success rate). Trained with 128 e","Best Optuna trial (v3 study, trial #28, 88.9% success rate). Trained with 128 en","Best Optuna trial (v3 study, trial #28, 88.9% success rate). Trained with 128 en","Best Optuna trial (v1 study, trial #100, 41.4% success rate). Trained with 128 e","Best Optuna trial (v1 study, trial #100, 41.4% success rate). Trained with 128 e","Best Optuna trial (v3 study, trial #28, 88.9% success rate). Trained with 128 en","Best Optuna trial (v3 study, trial #28, 88.9% success rate). Trained with 128 en","Best Optuna trial (v3 study, trial #28, 88.9% success rate). Trai